# Imbalanced Classification

The notebook keeps resampling and threshold choice inside the training-side discussion.

In [ ]:
from pathlib import Path
import sys

LESSON_REL = Path('phases/02-ml-fundamentals/17-imbalanced-data')
MODULE_FILE = 'imbalanced.py'
roots = [Path.cwd(), *Path.cwd().parents]
candidates = [Path.cwd() / LESSON_REL / 'code', Path.cwd() / 'code']
candidates.extend(root / LESSON_REL / 'code' for root in roots)
candidates.extend(root / 'code' for root in roots)
CODE = next((candidate.resolve() for candidate in candidates if (candidate / MODULE_FILE).is_file()), None)
if CODE is None:
    raise RuntimeError('Could not locate ' + str(LESSON_REL / 'code' / MODULE_FILE))
sys.path.insert(0, str(CODE))

## Build It

Run the next cell from the lesson directory; all values are local, deterministic fixtures.

In [ ]:
import numpy as np
import imbalanced as im

X, y = im.make_imbalanced_data(40, 8, seed=7)
split = 36
X_train, y_train = X[:split], y[:split]
X_over, y_over = im.random_oversample(X_train, y_train, seed=7)
minority = X_train[y_train == 1]
synthetic = im.smote(minority, k=min(5, len(minority)-1), n_synthetic=10, seed=7)
weights = im.compute_class_weights(y_train)
w, b = im.logistic_regression_weighted(X_train, y_train, weights, epochs=30)
probs = im.sigmoid(X[split:] @ w + b)
assert len(X_over) == len(y_over)
assert synthetic.shape == (10, 2)
assert np.isfinite(probs).all()
try:
    im.random_oversample(X_train, np.zeros(len(y_train), dtype=int))
except ValueError:
    pass
else:
    raise AssertionError('one-class resampling must be rejected')
try:
    im.class_weighted_loss(y_train, np.full(len(y_train), .5), np.zeros(len(y_train)))
except ValueError:
    pass
else:
    raise AssertionError('zero-total weights must be rejected')

## Exercises and Ship It

Pick a validation threshold with find_optimal_threshold rather than using test labels, then report prevalence, recall, precision, MCC, and the resampling boundary.

Record the observed shape/value and update the lesson output card. A notebook result is evidence for this fixture, not a production guarantee.